In [180]:
# load datasets
import pandas as pd
import matplotlib.pylab as plt
import numpy as np

df_sales = pd.read_csv('Sales.csv', encoding='latin1')                      # Cleaned
df_customers = pd.read_csv('Customers.csv', encoding='latin1')              # Cleaned
df_products = pd.read_csv('Products.csv', encoding='latin1')                # Cleaned
df_stores = pd.read_csv('Stores.csv', encoding='latin1')                    # Cleaned
df_fx = pd.read_csv('Exchange_Rates.csv', encoding='latin1')

# Data consistency checks and cleaning
1. **Sales** dataframe
* Converted ```Order Date``` and ```Delivery Date``` to datetime data type
* The sales dataset contains **nine** columns. ```Delivery Date``` contains **49719** number of empty fields representing **79%** of the total columns
* Sales orders span from ```2016-01-01``` to ```2021-02-20```
- Derrived columns
* Created ```order_month``` and ```order_year``` columns for better time series analysis
2. **Products** dataframe
* The products dataframe contains 2517 rows and 10 columns. This dataset does not contain any null/empty or duplicate values
* Removed **$** symbol from ```Unit Cost USD``` and ```Unit Price USD```
* After analysing the rest of the table I discovered that ```Unit Cost USD``` and ```Unit Price USD``` are object type. 
* Converted ```Unit Cost USD``` and ```Unit Price USD``` to integer float
3. Customers dataframe
* This dataset contains **15266** rows and **10** columns
* Found **10** empty values in the ```State Code``` column
* Converted customer birthday to datetime data type
4. Stores
* This dataset contains **67** rows and **5** columns 
* After the data checks it was found that one shop is missing ```Square Meters``` meassurements. No cleaning or data replacement was implemented in this column
* No duplicates were found


In [181]:
df_sales['Order Date'] = pd.to_datetime(df_sales['Order Date'])
df_sales['Delivery Date'] = pd.to_datetime(df_sales['Delivery Date'])
null_value_delivery_date = df_sales['Delivery Date'].isnull().mean() * 100

df_sales['order_month'] = df_sales['Order Date'].dt.to_period('M')
df_sales['order_year'] = df_sales['Order Date'].dt.year
df_sales.head()

,Order Number,Line Item,Order Date,Delivery Date,CustomerKey,StoreKey,ProductKey,Quantity,Currency Code,order_month,order_year
0,366000,1,2016-01-01,NaT,265598,10,1304,1,CAD,2016-01,2016
1,366001,1,2016-01-01,2016-01-13,1269051,0,1048,2,USD,2016-01,2016
2,366001,2,2016-01-01,2016-01-13,1269051,0,2007,1,USD,2016-01,2016
3,366002,1,2016-01-01,2016-01-12,266019,0,1106,7,CAD,2016-01,2016
4,366002,2,2016-01-01,2016-01-12,266019,0,373,1,CAD,2016-01,2016


In [182]:
for col in ['Unit Cost USD', 'Unit Price USD']:
    df_products[col] = (
        df_products[col]
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    
for col in ['Unit Cost USD', 'Unit Price USD']:
    df_products[col] = pd.to_numeric(df_products[col], errors='coerce')


In [183]:
df_customers.head()
df_customers['Birthday'] = pd.to_datetime(df_customers['Birthday'])

In [184]:
df_stores['Open Date'] = pd.to_datetime(df_stores['Open Date'])

In [185]:
# Joining products dataframe and sales
main_df = pd.merge(df_sales, df_products, how='left', on='ProductKey')


In [186]:
# del main_df['Currency Code']
# del main_df['SubcategoryKey']
# del main_df['CategoryKey']

main_df['Revenue'] = main_df['Quantity'] * main_df['Unit Price USD']
main_df['Profit'] = (main_df['Quantity'] * main_df['Unit Price USD']) - (main_df['Quantity'] * main_df['Unit Cost USD'])
# Cost per transaction
main_df['Cost'] = main_df['Quantity'] * main_df['Unit Cost USD']

# profit margin per product
main_df['Profit Margin'] = np.where(
    main_df['Revenue'] != 0,
    (main_df['Profit'] / main_df['Revenue']) * 100,
    0
)



In [187]:
# find out the main KPIs
from numerize import numerize
total_revenue = main_df['Revenue'].sum()
total_profit = main_df['Profit'].sum()
total_costs = main_df['Cost'].sum()
overall_profit_margin = (total_profit / total_revenue) * 100

total_volume = main_df['Quantity'].sum()


# year groups
year_groups = main_df.groupby('order_year')[['Revenue', 'Cost', 'Profit']].sum().reset_index()

# first year grouping
year_month = main_df.groupby(["order_year", "order_month"])[["Revenue", "Cost", "Profit"]].sum().reset_index()

# best profit grouping by revenue, costs and profit
top_products = main_df.groupby(['Product Name'])[['Revenue', 'Cost', 'Profit']].sum().sort_values('Profit', ascending=False).head(10).reset_index()



def plot_month_years(year):
    plt.figure(figsize=(6, 4))
    df_2016 = year_month[year_month['order_year'] == year]

    df_2016.plot(
        x="order_month",
        y=["Revenue", "Cost", "Profit"],
        kind="bar"
    )

    plt.grid(True)
    plt.legend()
    plt.title("2016 Monthly Performance")
    plt.show()

# join customers dataframe with main df on CustomerKey
main_df = pd.merge(main_df, df_customers, on='CustomerKey', how='left')

In [190]:
# top and lowest earning countries
top_five_revenue_countries = main_df.groupby(['Country'])[['Quantity', 'Revenue', 'Profit']].sum().sort_values('Revenue', ascending=False).reset_index().head(5)
lowest_five_revenue_countries = main_df.groupby(['Country'])[['Quantity', 'Revenue', 'Profit']].sum().sort_values('Revenue', ascending=True).reset_index().head(5)


top_five_revenue_cities = main_df.groupby(['City'])[['Quantity', 'Revenue', 'Profit']].sum().sort_values('Revenue', ascending=False).reset_index().head(5)
lowest_five_revenue_cities = main_df.groupby(['City'])[['Quantity', 'Revenue', 'Profit']].sum().sort_values('Revenue', ascending=True).reset_index().head(5)


print("Country exists:", 'Country' in main_df.columns)


Country exists: True
